# Nacimientos en Chile entre 2020 y 2023
## Fase 1 — Definición del Problema y Objetivos

**Grupo 2:** Cristian Hurtado, Jorge Altamirano, Renzo Vilchez y Luis Paredes.  
**Asignatura:** MCDI500 - Programación para la Ciencia de Datos.  
**Repositorio:** https://github.com/jorgealtamiranonavarrete-art/proyecto-grupo2

Este notebook define tres preguntas descriptivas, documenta la fuente y verifica las condiciones
de ejecución del proyecto. El análisis considera los cuatro años en conjunto. En F1 no se limpian,
imputan ni transforman los registros: esas tareas corresponden a F2.

Adaptación del notebook docente `S1_F1_Definicion.ipynb` de MCDI500, Universidad Andrés Bello.
Se conserva su organización en definición, entorno, estructura, módulos, fuente, Git,
validación, vinculación y documentación. El planteamiento se adapta a nacimientos.

### Ejecución
Guardar este archivo en `F1/notebooks/F1_Definicion.ipynb` dentro del repositorio clonado.
Seleccionar el entorno Python del proyecto en VS Code y ejecutar las celdas en orden.
Se propone Python 3.13; la versión efectivamente utilizada se registra al ejecutar.

El CSV esperado es `data/raw/Serie_Nacimientos_2020_2023.csv`. Si aún no está disponible,
la comprobación del archivo informa **por completar** y el resto de F1 continúa.
La dependencia de análisis de F1 es pandas; Jupyter o VS Code con ipykernel permiten ejecutarlo.

Este notebook genera únicamente su módulo `src/f1_utilidades.py` y documentación en `docs/f1/`.
No modifica el CSV, el README principal, `.gitignore` ni `requirements.txt`. Si el módulo existente
es distinto del propuesto, detiene su creación para evitar sobrescribir trabajo del equipo.
No crea commits ni sube cambios a GitHub.

In [59]:
import sys
import json
import platform
import hashlib
import subprocess
import shutil
import importlib.util
from importlib import metadata
from pathlib import Path
from datetime import date

try:
    import pandas as pd
except ImportError as error:
    raise ImportError("Falta pandas en el kernel seleccionado. Instálelo en el entorno del proyecto.") from error

print("F1 — fecha de ejecución:", date.today().isoformat())

F1 — fecha de ejecución: 2026-09-24


## 1. Definición del problema

Los registros individuales no permiten comparar directamente regiones de distinto tamaño ni meses
de distinta duración. El proyecto describirá la proporción regional de nacimientos de madres menores
de 20 años, el promedio diario por mes y la talla promedio por región. Sus resultados apoyarán la
selección de regiones y meses que destacar en las tablas y gráficos del informe académico.

Una fila representa un nacido vivo inscrito. No representa necesariamente una madre distinta ni
un parto único. Por ello, los resultados no se interpretarán como porcentajes de embarazos adolescentes.
El objetivo de aprendizaje es organizar y reproducir un proyecto con Python, notebooks, Git y GitHub.

In [60]:
PROYECTO = {'titulo': 'Nacimientos en Chile entre 2020 y 2023',
 'grupo': 'Grupo 2',
 'asignatura': 'MCDI500',
 'integrantes': ['Cristian Hurtado', 'Jorge Altamirano', 'Renzo Vilchez', 'Luis Paredes'],
 'repositorio': 'https://github.com/jorgealtamiranonavarrete-art/proyecto-grupo2',
 'problematica': 'Describir diferencias regionales en maternidad adolescente y talla al '
                 'nacer, y diferencias mensuales en la frecuencia diaria de nacimientos.',
 'objetivo_general': 'Describir la proporción regional de nacidos vivos de madres menores de '
                     '20 años, la distribución mensual del promedio diario de nacimientos y '
                     'la talla media regional en el período acumulado 2020–2023 mediante un '
                     'flujo reproducible.',
 'objetivos_especificos': ['Definir preguntas, indicadores, alcance y fuente en F1.',
                           'Verificar el entorno y documentar la estructura del repositorio '
                           'en F1.',
                           'Diagnosticar y preparar los datos en F2, evaluando el tratamiento '
                           'de los faltantes de talla.',
                           'Calcular y comparar los tres indicadores descriptivos en la fase '
                           'de análisis.',
                           'Comunicar los resultados y conservar evidencia de las '
                           'contribuciones del equipo.'],
 'preguntas': ['¿Qué regiones de residencia materna presentan mayor y menor porcentaje de '
               'nacidos vivos de madres menores de 20 años durante 2020–2023?',
               '¿Qué meses del calendario presentan mayor y menor promedio diario de '
               'nacimientos durante 2020–2023, considerando todas las edades maternas?',
               '¿Qué regiones de residencia materna presentan mayor y menor talla promedio de '
               'los recién nacidos durante 2020–2023, considerando todas las edades '
               'maternas?'],
 'criterios_exito': ['F1 y F2 pueden ejecutarse en el entorno documentado desde un kernel '
                     'reiniciado.',
                     'Las decisiones de preparación se justifican con conteos y '
                     'comparaciones.',
                     'Los tres integrantes mantienen contribuciones propias trazables en Git.',
                     'Los indicadores usan el período completo y denominadores documentados.'],
 'alcance_f1': {'incluye': ['Definición del proyecto',
                            'Documentación y reconocimiento de la fuente',
                            'Entorno, estructura y validación técnica'],
                'excluye': ['Limpieza e imputación',
                            'Transformación de observaciones',
                            'Cálculo de resultados finales']},
 'limitaciones': ['Los datos describen nacidos vivos registrados, no todos los embarazos.',
                  'No hay identificador de madre o parto para contar eventos únicos.',
                  'La región es la residencia de la madre, no necesariamente el lugar del '
                  'nacimiento.',
                  'El diseño es descriptivo, sin inferencia causal ni predicción.',
                  'La imputación de talla puede modificar medias y reducir la variabilidad.',
                  'La cobertura y el cierre del registro condicionan los resultados.']}

def presentar_proyecto(config):
    """Valida la definición mínima y muestra las preguntas del proyecto."""
    requeridas = ("titulo", "grupo", "integrantes", "objetivo_general", "preguntas")
    faltantes = [clave for clave in requeridas if not config.get(clave)]
    if faltantes:
        raise ValueError(f"Definición incompleta: {faltantes}")
    print(config["titulo"])
    print(config["objetivo_general"])
    for numero, pregunta in enumerate(config["preguntas"], 1):
        print(f"{numero}. {pregunta}")
    return True

presentar_proyecto(PROYECTO)

Nacimientos en Chile entre 2020 y 2023
Describir la proporción regional de nacidos vivos de madres menores de 20 años, la distribución mensual del promedio diario de nacimientos y la talla media regional en el período acumulado 2020–2023 mediante un flujo reproducible.
1. ¿Qué regiones de residencia materna presentan mayor y menor porcentaje de nacidos vivos de madres menores de 20 años durante 2020–2023?
2. ¿Qué meses del calendario presentan mayor y menor promedio diario de nacimientos durante 2020–2023, considerando todas las edades maternas?
3. ¿Qué regiones de residencia materna presentan mayor y menor talla promedio de los recién nacidos durante 2020–2023, considerando todas las edades maternas?


True

### Definiciones de los indicadores

| Pregunta | Numerador o medida | Denominador y alcance |
|---|---|---|
| Maternidad adolescente | NV de los grupos `MENORES 15 AÑOS` y `15 A 19 AÑOS`, multiplicados por 100 | Todos los NV de la misma región durante 2020–2023 |
| Promedio diario mensual | NV de cada mes calendario sumados en los cuatro años | Días de ese mes en los cuatro años: febrero 113, meses de 30 días 120 y meses de 31 días 124 |
| Talla media regional | Suma de tallas válidas, en centímetros, de cada región | Número de tallas válidas de esa región; comparar con los escenarios de imputación en F2 |

NV significa nacidos vivos. El total calendario es de 1.461 días. No se promediarán porcentajes
anuales ni se elaborarán rankings por año. Los empates se identificarán antes del redondeo.
Los 597 nulos iniciales de talla se evaluarán sin excluir esos nacimientos de las otras dos preguntas.

### Condiciones de reutilización y redistribución

El dataset proviene del portal Datos Abiertos del DEIS/MINSAL:  
https://deis.minsal.cl/#datosabiertos

Se utilizará con fines académicos para analizar descriptivamente los nacimientos de 2020 a 2023. La descarga se realizó el 15/09/2026. Se mantendrá la atribución a DEIS/MINSAL y el archivo se conserva en `data/raw/` para permitir la reproducción del análisis.

## 2. Verificación del entorno
Se registra el intérprete real y las versiones instaladas. Detectar un entorno aislado no demuestra
por sí solo que sea el acordado: cada integrante debe seleccionar y verificar el kernel del proyecto.
Este análisis no necesita aleatoriedad; por tanto, no se fija una semilla sin uso. Si F2 incorpora un
procedimiento aleatorio, deberá registrar su semilla.

In [47]:
def verificar_entorno():
    """Devuelve versiones reales y condiciones del intérprete, sin instalar paquetes."""
    versiones = {}
    for paquete in ("pandas", "numpy", "matplotlib", "ipykernel", "jupyterlab"):
        try:
            versiones[paquete] = metadata.version(paquete)
        except metadata.PackageNotFoundError:
            versiones[paquete] = None
    return {
        "python": platform.python_version(), "interprete": sys.executable,
        "sistema": platform.system(),
        "entorno_virtual_detectado": sys.prefix != sys.base_prefix or (Path(sys.prefix) / "conda-meta").is_dir(),
        "versiones": versiones,
        "confirmacion_kernel_equipo": "por completar: confirmar el entorno acordado y reproducir con otro integrante",
    }

ENTORNO = verificar_entorno()
print(json.dumps(ENTORNO, indent=2, ensure_ascii=False))

{
  "python": "3.13.15",
  "interprete": "c:\\Users\\vilch\\proyecto-grupo2\\.venv313\\Scripts\\python.exe",
  "sistema": "Windows",
  "entorno_virtual_detectado": true,
  "versiones": {
    "pandas": "2.2.3",
    "numpy": "2.1.3",
    "matplotlib": "3.10.0",
    "ipykernel": "6.29.5",
    "jupyterlab": "4.3.4"
  },
  "confirmacion_kernel_equipo": "por completar: confirmar el entorno acordado y reproducir con otro integrante"
}


## 3. Estructura y archivos del repositorio
Las fases comparten `data/raw/` y `data/processed/`. El archivo original se conserva sin cambios.
La raíz se busca desde la carpeta de ejecución hacia sus padres, para poder ejecutar desde
`F1/notebooks/` o desde la raíz. Si no se identifica el proyecto, se informa el problema en lugar
de crear carpetas en una ubicación arbitraria.

In [48]:
def encontrar_raiz(inicio):
    """Localiza una raíz con carpetas F1, F2 y data; falla si no existe."""
    inicio = Path(inicio).resolve()
    for candidato in (inicio, *inicio.parents):
        if all((candidato / nombre).is_dir() for nombre in ("F1", "F2", "data")):
            return candidato
    raise FileNotFoundError("Abra el notebook dentro del repositorio que contiene F1/, F2/ y data/.")

RAIZ = encontrar_raiz(Path.cwd())
DIR_DOCS = RAIZ / "docs" / "f1"
DIR_SRC = RAIZ / "src"
for relativo in ("data/raw", "data/processed", "docs/f1", "src", "F1/notebooks", "F2", "outputs"):
    (RAIZ / relativo).mkdir(parents=True, exist_ok=True)
ARCHIVO = RAIZ / "data" / "raw" / "Serie_Nacimientos_2020_2023.csv"
print("Raíz:", RAIZ)
for relativo in ("data/raw", "data/processed", "docs/f1", "src", "F1/notebooks", "F2", "outputs"):
    print(relativo + "/")

ESTADO_ARCHIVOS = {}
for nombre in ("README.md", ".gitignore", "requirements.txt"):
    ruta = RAIZ / nombre
    estado = "presente con contenido" if ruta.exists() and ruta.read_text(encoding="utf-8").strip() else "por completar"
    ESTADO_ARCHIVOS[nombre] = estado
print(ESTADO_ARCHIVOS)

Raíz: C:\Users\vilch\proyecto-grupo2
data/raw/
data/processed/
docs/f1/
src/
F1/notebooks/
F2/
outputs/
{'README.md': 'por completar', '.gitignore': 'presente con contenido', 'requirements.txt': 'presente con contenido'}


### Dependencias y archivos compartidos
El README principal debe describir el proyecto, la descarga de datos, las dependencias y el orden de ejecución.
`.gitignore` debe excluir el entorno, las cachés y las credenciales. F1 genera en `docs/f1/` un borrador
de README y una lista de versiones observadas; el equipo revisará su integración sin reemplazar archivos
compartidos automáticamente. Las versiones de un computador no constituyen evidencia de reproducción en otro.

## 4. Módulo reutilizable y pruebas
Se utiliza un módulo pequeño para verificar columnas obligatorias y calcular la huella del archivo.
Son funciones útiles para F1 y F2 que no alteran los datos. Se incluyen pruebas normales, de límite y
de excepción; una excepción esperada que no ocurra hará fallar la prueba.

In [49]:
CODIGO_MODULO = '"""Verificaciones compartidas del proyecto de nacimientos, Grupo 2."""\nimport hashlib\nfrom pathlib import Path\n\nclass ProyectoError(ValueError):\n    """Entrada incompatible con las verificaciones del proyecto."""\n\ndef verificar_columnas(disponibles, requeridas):\n    """Devuelve True si están las columnas requeridas; no modifica la entrada."""\n    if not isinstance(disponibles, (list, tuple)) or not isinstance(requeridas, (list, tuple)):\n        raise ProyectoError("Las columnas deben proporcionarse como listas o tuplas.")\n    if not all(isinstance(c, str) for c in [*disponibles, *requeridas]):\n        raise ProyectoError("Cada nombre de columna debe ser texto.")\n    faltantes = sorted(set(requeridas) - set(disponibles))\n    if faltantes:\n        raise ProyectoError(f"Columnas ausentes: {faltantes}")\n    return True\n\ndef huella_sha256(ruta):\n    """Calcula SHA-256 por bloques sin alterar el archivo."""\n    digest = hashlib.sha256()\n    with Path(ruta).open("rb") as archivo:\n        for bloque in iter(lambda: archivo.read(1024 * 1024), b""):\n            digest.update(bloque)\n    return digest.hexdigest()\n'

ruta_modulo = DIR_SRC / "f1_utilidades.py"
if ruta_modulo.exists() and ruta_modulo.read_text(encoding="utf-8") != CODIGO_MODULO:
    raise FileExistsError("src/f1_utilidades.py contiene otra versión. Revise las diferencias antes de continuar.")
if not ruta_modulo.exists():
    ruta_modulo.write_text(CODIGO_MODULO, encoding="utf-8")
spec = importlib.util.spec_from_file_location("f1_utilidades", ruta_modulo)
utilidades = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utilidades)
print("Módulo disponible:", ruta_modulo.relative_to(RAIZ))

Módulo disponible: src\f1_utilidades.py


In [50]:
def esperar_error(tipo, funcion, *args):
    """Comprueba que una entrada inválida produzca la excepción prevista."""
    try:
        funcion(*args)
    except tipo:
        return True
    raise AssertionError(f"Se esperaba {tipo.__name__} y no ocurrió.")

assert utilidades.verificar_columnas(["ANO_NAC", "TALLA"], ["TALLA"])
assert utilidades.verificar_columnas([], [])
assert esperar_error(utilidades.ProyectoError, utilidades.verificar_columnas, ["ANO_NAC"], ["TALLA"])
assert esperar_error(utilidades.ProyectoError, utilidades.verificar_columnas, [42], ["TALLA"])
assert esperar_error(ValueError, presentar_proyecto, {})
assert utilidades.huella_sha256(ruta_modulo) == hashlib.sha256(ruta_modulo.read_bytes()).hexdigest()
PRUEBAS_MODULO = True
print("Seis comprobaciones superadas: casos normales, límite, excepciones y huella.")

Seis comprobaciones superadas: casos normales, límite, excepciones y huella.


## 5. Fuente y diccionario de variables
Se utilizará exclusivamente `Serie_Nacimientos_2020_2023.csv`, publicado por DEIS/MINSAL.
`Fichas DA Nacimientos.xlsx` aporta documentación y no constituye un segundo dataset.
La unidad de observación es un nacido vivo inscrito; la región corresponde a la residencia materna.

**Referencia:** Departamento de Estadísticas e Información de Salud. (s. f.). *Serie de nacimientos
2020–2023 y Fichas DA Nacimientos* [Conjunto de datos y documentación]. Ministerio de Salud de Chile.
https://deis.minsal.cl/#datosabiertos

**Por completar:** fecha original de descarga, enlace directo o identificación de la versión publicada
y condiciones específicas de reutilización. El acceso público no demuestra por sí mismo una licencia abierta.
La fecha de ejecución de este notebook no sustituye la fecha de descarga.

La ficha y el CSV presentan diferencias de nombres y porcentajes: por ejemplo, `PESO` en la ficha
corresponde a `RANGO_PESO` en el archivo. Se usan los nombres reales del CSV y sus conteos observados.

In [51]:
FICHA = {
    "titulo": "Serie de nacimientos 2020–2023",
    "fuente": "DEIS / Ministerio de Salud de Chile",
    "url": "https://deis.minsal.cl/#datosabiertos",
    "archivo": ARCHIVO.name, "separador": ";", "codificacion": "utf-8",
    "unidad_observacion": "Un nacido vivo inscrito",
    "periodo": [2020, 2021, 2022, 2023],
    "filas_referencia": 735611, "columnas_referencia": 25,
    "talla_nulos_referencia": 597,
    "fecha_descarga": "15 de septiembre de 2026",
    "version_publicada_o_url_directa": "https://repositoriodeis.minsal.cl/DatosAbiertos/VITALES/NACIMIENTOS/Serie_Nacimientos_2020_2023.zip",
    "licencia": "Datos públicos Abiertos (Ley N° 20.285 sobre Acceso a la Información Pública)",
}

In [52]:
DICCIONARIO = [('MES_NAC', 'temporal', 'Mes de nacimiento', 'P2'),
 ('ANO_NAC', 'temporal', 'Año de nacimiento', 'P1, P2, P3'),
 ('SEXO', 'nominal', 'Sexo registrado del nacido vivo', 'Contexto'),
 ('TIPO_PARTO', 'nominal', 'Tipo de parto según código', 'Contexto'),
 ('TIPO_ATEN', 'nominal', 'Profesional o tipo de atención', 'Contexto'),
 ('PARTO_LOCAL', 'nominal', 'Lugar del nacimiento', 'Contexto'),
 ('SEMANAS', 'duración registrada en semanas', 'Semanas de gestación', 'Contexto'),
 ('RANGO_PESO', 'ordinal', 'Intervalo de peso, no peso exacto', 'Contexto'),
 ('TALLA', 'cuantitativa continua', 'Talla al nacer en centímetros', 'P3'),
 ('GRUPO_ETARIO_PADRE', 'ordinal', 'Intervalo de edad paterna', 'Contexto'),
 ('CURSO_PADRE', 'discreta', 'Último curso de instrucción del padre', 'Contexto'),
 ('NIVEL_PADRE',
  'categoría educativa',
  'Nivel educacional del padre; revisar orden de códigos',
  'Calidad de la base'),
 ('ACTIV_PADRE', 'nominal', 'Actividad del padre', 'Calidad de la base'),
 ('OCUPA_PADRE', 'nominal', 'Ocupación paterna condicionada a actividad', 'Contexto'),
 ('CATEG_PADRE', 'nominal', 'Categoría ocupacional paterna', 'Contexto'),
 ('GRUPO_ETARIO_MADRE', 'ordinal', 'Intervalo de edad materna', 'P1'),
 ('EST_CIV_MADRE', 'nominal', 'Estado civil materno', 'Contexto'),
 ('CURSO_MADRE', 'discreta', 'Último curso de instrucción de la madre', 'Contexto'),
 ('NIVEL_MADRE',
  'categoría educativa',
  'Nivel educacional materno; revisar orden de códigos',
  'Contexto'),
 ('ACTIV_MADRE', 'nominal', 'Actividad materna', 'Contexto'),
 ('OCUPA_MADRE', 'nominal', 'Ocupación materna condicionada a actividad', 'Contexto'),
 ('CATEG_MADRE', 'nominal', 'Categoría ocupacional materna', 'Contexto'),
 ('NACIONALIDAD_MADRE', 'nominal', 'Categoría de nacionalidad materna', 'Contexto'),
 ('REGION_RESIDENCIA', 'nominal', 'Código regional de residencia de la madre', 'P1, P3'),
 ('GLOSA_REGION_RESIDENCIA', 'nominal', 'Nombre regional de residencia de la madre', 'P1, P3')]
diccionario = pd.DataFrame(DICCIONARIO, columns=["variable", "rol", "descripcion", "uso"])
print(diccionario.to_string(index=False))

               variable                            rol                                           descripcion                uso
                MES_NAC                       temporal                                     Mes de nacimiento                 P2
                ANO_NAC                       temporal                                     Año de nacimiento         P1, P2, P3
                   SEXO                        nominal                       Sexo registrado del nacido vivo           Contexto
             TIPO_PARTO                        nominal                            Tipo de parto según código           Contexto
              TIPO_ATEN                        nominal                        Profesional o tipo de atención           Contexto
            PARTO_LOCAL                        nominal                                  Lugar del nacimiento           Contexto
                SEMANAS duración registrada en semanas                                  Semanas de gesta

### Reconocimiento del archivo sin limpieza
Se comprueban dimensiones, columnas, años y nulos. Los conteos de F1 reconocen la fuente y permiten
evaluar su selección. F2 profundizará en códigos especiales, rangos, filas iguales y tratamientos.
Un código “ignorado” no es automáticamente un nulo: requiere interpretación del diccionario.

In [53]:
def reconocer_archivo(ruta, ficha, columnas):
    """Lee el CSV y resume estructura y nulos; no limpia ni guarda observaciones."""
    if not ruta.exists():
        return {"estado": "por completar", "motivo": f"Colocar {ruta.name} en data/raw/"}
    datos = pd.read_csv(ruta, sep=ficha["separador"], encoding=ficha["codificacion"], low_memory=False)
    utilidades.verificar_columnas(list(datos.columns), columnas)
    if len(datos) == 0:
        raise ValueError("El CSV no contiene registros.")
    nulos = datos.isna().sum()
    return {
        "estado": "archivo reconocido", "filas": len(datos), "columnas": len(datos.columns),
        "columnas_exactas": set(datos.columns) == set(columnas),
        "dimensiones_coinciden": datos.shape == (ficha["filas_referencia"], ficha["columnas_referencia"]),
        "anios": sorted(int(x) for x in datos["ANO_NAC"].dropna().unique()),
        "regiones": int(datos["REGION_RESIDENCIA"].nunique()),
        "nulos": {k: int(v) for k, v in nulos.items()},
        "porcentaje_nulos": {k: float(v / len(datos) * 100) for k, v in nulos.items()},
        "cardinalidades": {k: int(v) for k, v in datos.nunique().items()},
        "bytes": ruta.stat().st_size, "sha256": utilidades.huella_sha256(ruta),
    }

DIAGNOSTICO = reconocer_archivo(ARCHIVO, FICHA, diccionario["variable"].tolist())
print(json.dumps(DIAGNOSTICO, indent=2, ensure_ascii=False))

{
  "estado": "archivo reconocido",
  "filas": 735611,
  "columnas": 25,
  "columnas_exactas": true,
  "dimensiones_coinciden": true,
  "anios": [
    2020,
    2021,
    2022,
    2023
  ],
  "regiones": 16,
  "nulos": {
    "MES_NAC": 0,
    "ANO_NAC": 0,
    "SEXO": 0,
    "TIPO_PARTO": 0,
    "TIPO_ATEN": 0,
    "PARTO_LOCAL": 0,
    "SEMANAS": 595,
    "RANGO_PESO": 0,
    "TALLA": 597,
    "GRUPO_ETARIO_PADRE": 0,
    "CURSO_PADRE": 0,
    "NIVEL_PADRE": 50737,
    "ACTIV_PADRE": 60962,
    "OCUPA_PADRE": 0,
    "CATEG_PADRE": 0,
    "GRUPO_ETARIO_MADRE": 0,
    "EST_CIV_MADRE": 0,
    "CURSO_MADRE": 0,
    "NIVEL_MADRE": 0,
    "ACTIV_MADRE": 0,
    "OCUPA_MADRE": 1,
    "CATEG_MADRE": 0,
    "NACIONALIDAD_MADRE": 0,
    "REGION_RESIDENCIA": 0,
    "GLOSA_REGION_RESIDENCIA": 0
  },
  "porcentaje_nulos": {
    "MES_NAC": 0.0,
    "ANO_NAC": 0.0,
    "SEXO": 0.0,
    "TIPO_PARTO": 0.0,
    "TIPO_ATEN": 0.0,
    "PARTO_LOCAL": 0.0,
    "SEMANAS": 0.08088514173931603,
    "RANGO_PES

### Evaluación de criterios del ejemplo docente
La sección 5.2 del ejemplo pide alguna variable con al menos 1 % de nulos y ninguna con más del 60 %.
La revisión previa del CSV encontró 6,90 % en `NIVEL_PADRE` y 8,29 % en `ACTIV_PADRE`.
`TALLA` tiene 0,081 %: mantenemos su pregunta y evaluaremos su imputación en F2.

El ejemplo también pide dos variables continuas, una fecha y una variable de texto o alta cardinalidad.
No declaramos esos criterios cumplidos automáticamente: el peso está agrupado, la fecha disponible tiene
solo año y mes y la glosa regional es texto con 16 categorías. **Por completar:** confirmar con el profesor
cómo aplica esos criterios al alcance descriptivo. No se inventarán columnas o errores para aparentar cumplimiento.

In [54]:
def evaluar_criterios(diagnostico):
    """Separa comprobaciones medibles de criterios que requieren interpretación docente."""
    presente = diagnostico.get("estado") == "archivo reconocido"
    filas = []
    verificaciones = {
        "Al menos 2.000 filas": presente and diagnostico.get("filas", 0) >= 2000,
        "Al menos 12 columnas": presente and diagnostico.get("columnas", 0) >= 12,
        "Alguna variable con al menos 1 % de nulos": presente and any(v >= 1 for v in diagnostico.get("porcentaje_nulos", {}).values()),
        "Ninguna variable con más de 60 % de nulos": presente and all(v <= 60 for v in diagnostico.get("porcentaje_nulos", {}).values()),
    }
    for criterio, cumple in verificaciones.items():
        filas.append((criterio, "cumple" if cumple else ("no cumple" if presente else "por completar")))
    filas.extend([
        ("Numérica discreta", "identificada: CURSO_PADRE y CURSO_MADRE; verificar códigos en F2"),
        ("Dos nominales y una ordinal o binaria", "identificadas: región, sexo y grupo etario"),
        ("Dos numéricas continuas", "por completar: TALLA y criterio aplicado a SEMANAS; peso agrupado"),
        ("Fecha", "por completar: confirmar suficiencia de año y mes, sin día"),
        ("Texto o alta cardinalidad", "por completar: confirmar aceptación de glosa regional, 16 categorías"),
    ])
    return pd.DataFrame(filas, columns=["criterio", "estado"])

evaluacion = evaluar_criterios(DIAGNOSTICO)
print(evaluacion.to_string(index=False))

                                 criterio                                                               estado
                     Al menos 2.000 filas                                                               cumple
                     Al menos 12 columnas                                                               cumple
Alguna variable con al menos 1 % de nulos                                                               cumple
Ninguna variable con más de 60 % de nulos                                                               cumple
                        Numérica discreta     identificada: CURSO_PADRE y CURSO_MADRE; verificar códigos en F2
    Dos nominales y una ordinal o binaria                           identificadas: región, sexo y grupo etario
                  Dos numéricas continuas    por completar: TALLA y criterio aplicado a SEMANAS; peso agrupado
                                    Fecha           por completar: confirmar suficiencia de año y mes, sin día
 

## 6. Control de versiones y colaboración
Jorge es propietario del repositorio. Cada integrante registra sus cambios con commits propios y trabaja
en una rama. El pull request se somete a revisión de código y resultados por otro integrante antes de integrarse.
Las consultas siguientes son locales: no descargan cambios ni verifican permisos actuales en GitHub.

| Integrante | Responsabilidad acordada |
|---|---|
| Cristian Hurtado | Planteamiento, mapa, elaboración inicial de F1 para revisión de Jorge, F2 e informe |
| Jorge Altamirano | Revisión y desarrollo de F1, repositorio, integración e informe |
| Renzo Vilchez | Colaboración en F2 y revisión del trabajo, con apoyo para Python |

La cuenta de Cristian se ha identificado como `churtado82`; la de Jorge es `jorgealtamiranonavarrete-art` y la de Renzo es `renzovilchez02`.

In [55]:
def consultar_git(raiz, argumentos):
    """Ejecuta una consulta Git local y devuelve estado y salida."""
    ejecutable = shutil.which("git")
    if ejecutable is None:
        return {"ok": False, "salida": "por completar: Git no disponible en PATH"}
    try:
        proceso = subprocess.run([ejecutable, "-C", str(raiz), *argumentos], capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=20)
        return {"ok": proceso.returncode == 0, "salida": proceso.stdout.strip() or proceso.stderr.strip()}
    except (OSError, subprocess.SubprocessError) as error:
        return {"ok": False, "salida": str(error)}

GIT = {nombre: consultar_git(RAIZ, args) for nombre, args in {
    "rama": ["branch", "--show-current"],
    "ultimo_commit": ["log", "-1", "--format=%h %s"],
    "estado": ["status", "--short"],
}.items()}
for nombre, resultado in GIT.items():
    print(nombre, ":", resultado["salida"] or "sin cambios")

rama : main
ultimo_commit : 2680659 Merge branch 'main' of https://github.com/jorgealtamiranonavarrete-art/proyecto-grupo2
estado : ?? src/__pycache__/f1_utilidades.cpython-313.pyc


## 7. Validación técnica de F1
Las pruebas distinguen la integridad del notebook de la preparación completa de la entrega.
Que el código se ejecute no demuestra que el equipo haya completado los pendientes o reproducido el entorno.

In [56]:
def validar_fase1(config, diccionario, diagnostico):
    """Valida coherencia estructural sin convertir pendientes en comprobaciones exitosas."""
    assert len(config["preguntas"]) == 3
    assert len(config["integrantes"]) == 3
    assert len(diccionario) == 25 and diccionario["variable"].is_unique
    assert PRUEBAS_MODULO
    if diagnostico["estado"] == "archivo reconocido":
        assert diagnostico["columnas_exactas"], "Revisar el diccionario frente al CSV."
        assert diagnostico["dimensiones_coinciden"], "La versión del CSV difiere de la referencia."
        assert diagnostico["anios"] == [2020, 2021, 2022, 2023]
        assert diagnostico["regiones"] == 16
    return {"integridad_f1": "verificada", "archivo_disponible": diagnostico["estado"] == "archivo reconocido",
            "entrega_completa": False, "motivo": "Revisar lista de pendientes y reproducir en el entorno del equipo."}

VALIDACION = validar_fase1(PROYECTO, diccionario, DIAGNOSTICO)
print(VALIDACION)

{'integridad_f1': 'verificada', 'archivo_disponible': True, 'entrega_completa': False, 'motivo': 'Revisar lista de pendientes y reproducir en el entorno del equipo.'}


## 8. Vinculación con el mapa conceptual
La tabla distingue lo implementado en F1 de lo planificado para F2 y las fases posteriores.
Los resultados de Git y del entorno dependen del equipo donde se ejecute este notebook.

In [57]:
vinculacion = pd.DataFrame([
    ("Problema e indicadores", "Sección 1", "Definición y tres preguntas"),
    ("Fuente y población", "Sección 5", "Ficha, diccionario y reconocimiento si existe el CSV"),
    ("Entorno reproducible", "Sección 2", "Versiones observadas; reproducción del equipo por completar"),
    ("Estructura y documentación", "Secciones 3 y 9", "Rutas y documentos propios de F1"),
    ("Módulos y validación", "Secciones 4 y 7", "Funciones y comprobaciones ejecutables"),
    ("Git y revisión", "Sección 6", "Consultas locales; PR y revisión por completar"),
    ("Preparación de talla", "F2 pendiente", "Comparar sin imputar, mediana general y mediana por región"),
    ("Análisis y comunicación", "Fases posteriores", "Tablas y gráficos de los tres indicadores pendientes"),
], columns=["concepto", "ubicacion", "evidencia_o_estado"])
print(vinculacion.to_string(index=False))

                  concepto         ubicacion                                          evidencia_o_estado
    Problema e indicadores         Sección 1                                 Definición y tres preguntas
        Fuente y población         Sección 5        Ficha, diccionario y reconocimiento si existe el CSV
      Entorno reproducible         Sección 2 Versiones observadas; reproducción del equipo por completar
Estructura y documentación   Secciones 3 y 9                            Rutas y documentos propios de F1
      Módulos y validación   Secciones 4 y 7                      Funciones y comprobaciones ejecutables
            Git y revisión         Sección 6              Consultas locales; PR y revisión por completar
      Preparación de talla      F2 pendiente  Comparar sin imputar, mediana general y mediana por región
   Análisis y comunicación Fases posteriores        Tablas y gráficos de los tres indicadores pendientes


## 9. Documentación y pendientes
Se exportan los documentos propios de F1 a `docs/f1/`. Su regeneración reemplaza estas salidas,
no los archivos originales ni la documentación principal del equipo. Los pendientes se revisarán
antes de la entrega y se actualizarán con evidencia, sin inventar fechas, permisos o resultados.

In [62]:
PENDIENTES=[]
if DIAGNOSTICO["estado"] != "archivo reconocido":
    PENDIENTES.insert(0, "por completar: ubicar el CSV original en data/raw/ y ejecutar su reconocimiento")
for pendiente in PENDIENTES:
    print("-", pendiente)

for nombre, objeto in {
    "proyecto.json": PROYECTO, "ficha.json": FICHA,
    "reconocimiento.json": DIAGNOSTICO, "validacion.json": VALIDACION,
    "pendientes.json": PENDIENTES,
    "entorno.json": {**ENTORNO, "fecha_ejecucion": date.today().isoformat()},
}.items():
    (DIR_DOCS / nombre).write_text(json.dumps(objeto, indent=2, ensure_ascii=False), encoding="utf-8")
for nombre, tabla in {
    "diccionario_variables.csv": diccionario,
    "evaluacion_criterios.csv": evaluacion,
    "vinculacion_mapa.csv": vinculacion,
}.items():
    tabla.to_csv(DIR_DOCS / nombre, index=False)

lineas = [f"{paquete}=={version}" for paquete, version in ENTORNO["versiones"].items() if version]
(DIR_DOCS / "dependencias_observadas.txt").write_text("\n".join(lineas) + "\n", encoding="utf-8")
readme = f"""# {PROYECTO['titulo']}
Grupo 2: {', '.join(PROYECTO['integrantes'])}.
Repositorio: {PROYECTO['repositorio']}

## Alcance
Análisis descriptivo acumulado 2020–2023 de maternidad adolescente, frecuencia diaria mensual y talla regional.

## Datos
Fuente: {FICHA['url']}
Ubicación local: data/raw/{ARCHIVO.name}
La ficha Excel es documentación complementaria. Condiciones de reutilización: por completar.

## Ejecución
Crear un entorno del proyecto e instalar las dependencias acordadas en requirements.txt.
Seleccionar ese kernel en VS Code. Ejecutar primero F1/notebooks/F1_Definicion.ipynb.
F2: por completar. Python propuesto: 3.13; versión observada: {ENTORNO['python']}.
Las versiones detectadas están en docs/f1/dependencias_observadas.txt y deben revisarse antes de integrarse.

## Estructura
data/raw/: originales. data/processed/: preparación F2. src/: funciones.
F1/notebooks/: definición. F2/: preparación. docs/f1/: documentación F1. outputs/: resultados.

## Colaboración
Cada integrante trabaja en una rama, registra sus cambios y solicita revisión mediante pull request.
Ejemplos de mensajes: docs: define preguntas; chore: configura entorno; feat: verifica columnas.
"""
(DIR_DOCS / "README_propuesta.md").write_text(readme, encoding="utf-8")
print("Documentación F1 guardada en", DIR_DOCS.relative_to(RAIZ))

Documentación F1 guardada en docs\f1


## 10. Conclusiones y reflexión técnica
**Por completar por el grupo después de ejecutar y revisar F1.** Las siguientes ideas son propuestas
de redacción; no afirman que ya se hayan obtenido resultados regionales o mensuales.

1. La definición de nacido vivo como unidad de observación permite delimitar correctamente el proyecto:
   sus indicadores no representan el número de madres distintas ni todos los embarazos adolescentes.
2. El uso de denominadores regionales y días calendario hace que las comparaciones respondan a las
   preguntas planteadas, sin depender únicamente del tamaño regional o de la duración del mes.
3. La presencia de faltantes de talla requiere comparar tratamientos y documentar su efecto; una
   proporción pequeña de nulos no determina por sí sola que deban imputarse o descartarse.
4. La documentación de la fuente, las versiones y las rutas establece condiciones verificables para
   reproducir el proyecto. La reproducción por otro integrante todavía debe demostrarse.

**Reflexión propia por completar:** ¿qué dificultad real encontramos?, ¿qué decisión tomamos para
resolverla?, ¿qué comprobación demuestra que funcionó? Agregar experiencias efectivamente ocurridas.

## 11. Verificación antes de entregar
- Ejecutar todas las celdas desde un kernel reiniciado y revisar las salidas.
- Colocar el CSV en la ruta acordada y confirmar la huella de la versión utilizada.
- Resolver o explicar los pendientes de selección del dataset y licencia.
- Integrar README y dependencias comprobadas, sin sobrescribir cambios de compañeros.
- Revisar que F1 no contenga limpieza o imputación de registros.
- Confirmar que el notebook, mapa e informe coincidan en preguntas, período y repositorio.
- Conservar evidencia real de commits y revisión; no reemplazarla por afirmaciones escritas.
- Completar la reflexión con experiencias y resultados de ejecución del grupo.

### Fuentes
- Universidad Andrés Bello. (s. f.). *S1_F1_Definicion.ipynb* [Notebook docente de MCDI500].
- Universidad Andrés Bello. (s. f.). *Guía de desarrollo de la Evaluación Sumativa 1 Fases 1 y 2* [Material docente].
- DEIS/MINSAL. (s. f.). *Serie de nacimientos 2020–2023 y Fichas DA Nacimientos*.
  https://deis.minsal.cl/#datosabiertos